# Task 3 — Marketing Funnel & Conversion Performance Analysis

## 1. Data Loading & Sampling

The raw dataset (`2019-Oct.csv`) contains ~42.4M e-commerce events from October 2019.
Loading the full file is impractical, so we sample the **first 7 days** to preserve
complete sessions while keeping memory manageable.

In [1]:
import pandas as pd

chunks = []
for chunk in pd.read_csv('../data/2019-Oct.csv', chunksize=1000000, parse_dates=["event_time"]):
    chunk = chunk[chunk["event_time"] < "2019-10-08"]
    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
print(f"Loaded {len(df):,} rows")

Loaded 8,829,315 rows


In [2]:
# Save sample df
df.to_parquet("../data/sample_first_week.parquet", index=False)

## 2. Initial Inspection

Basic shape, dtypes, and uniqueness checks on the sampled week.

In [3]:
import sys
sys.path.append("..")

import pandas as pd
from utilities.cleaning import report, check_categories

# Load sample df
df = pd.read_parquet("../data/sample_first_week.parquet")

%load_ext autoreload
%autoreload 2

In [4]:
report(df)

Total rows: 8829315, Total columns: 9

Duplicate rows: 5712

Missing values:
category_code: 2784814 missing values
brand: 1212750 missing values
user_session: 1 missing values

The first and last 5 rows of the dataset:
                 event_time event_type  product_id          category_id  \
0 2019-10-01 00:00:00+00:00       view    44600062  2103807459595387724   
1 2019-10-01 00:00:00+00:00       view     3900821  2053013552326770905   
2 2019-10-01 00:00:01+00:00       view    17200506  2053013559792632471   
3 2019-10-01 00:00:01+00:00       view     1307067  2053013558920217191   
4 2019-10-01 00:00:04+00:00       view     1004237  2053013555631882655   

                         category_code     brand    price    user_id  \
0                                  NaN  shiseido    35.79  541312140   
1  appliances.environment.water_heater      aqua    33.20  554748717   
2           furniture.living_room.sofa       NaN   543.10  519107250   
3                   computers.notebook    

In [6]:
check_categories(df)

In [7]:
print(f"Unique users:    {df['user_id'].nunique():,}")
print(f"Unique sessions: {df['user_session'].nunique():,}")
print(f"Unique products: {df['product_id'].nunique():,}")
print(f"Date range:      {df['event_time'].min()} → {df['event_time'].max()}")

Unique users:    957,543
Unique sessions: 1,877,365
Unique products: 118,333
Date range:      2019-10-01 00:00:00+00:00 → 2019-10-07 23:59:59+00:00


## 3. Missing Value Analysis

Three columns have nulls: `category_code` (31.5%), `brand` (13.7%), and `user_session`
(1 row). Before deciding how to handle them, we diagnose whether the missingness is
random or not.

In [8]:
# Is category_id also missing?
print(df["category_id"].isnull().sum())

0


In [9]:
# Does missingness vary by event type?
print(df.groupby("event_type")["category_code"].apply(lambda x: x.isnull().mean()))

event_type
cart        0.093291
purchase    0.222831
view        0.321842
Name: category_code, dtype: float64


In [10]:
# Does missingness vary by brand?
brand_counts = df["brand"].value_counts()
big_brands = brand_counts[brand_counts > 1000].index

brand_code_miss = (
    df[df['brand'].isin(big_brands)].groupby('brand')['category_code'].apply(
        lambda x: x.isnull().mean()).sort_values(ascending=False)
)

print(brand_code_miss.head(10))   # worst 10
print(brand_code_miss.tail(10))   # best 10

brand
aeroforce    1.0
lucente      1.0
luminarc     1.0
marshal      1.0
matador      1.0
mart         1.0
maxtrek      1.0
maxxis       1.0
triangle     1.0
mercury      1.0
Name: category_code, dtype: float64
brand
axis        0.0
babytime    0.0
xerox       0.0
turboair    0.0
turbo       0.0
armani      0.0
adamex      0.0
yasin       0.0
zlatek      0.0
zte         0.0
Name: category_code, dtype: float64


In [11]:
mapping_check = (
    df.dropna(subset=["category_code"])
    .groupby("category_id")["category_code"].nunique()
)
print(f'category_ids with a code: {len(mapping_check):,}')
print(f'category_ids with exactly one code: {(mapping_check == 1).sum():,}')
print(f'category_ids with multiple codes: {(mapping_check > 1).sum():,}')

category_ids with a code: 235
category_ids with exactly one code: 235
category_ids with multiple codes: 0


In [12]:
ids_with_code = set(df.dropna(subset=['category_code'])['category_id'])
ids_in_null_rows = set(df[df['category_code'].isnull()]['category_id'])

unrecoverable = ids_in_null_rows - ids_with_code
recoverable = ids_in_null_rows & ids_with_code

print(f'Distinct category_ids in null-code rows: {len(ids_in_null_rows):,}')
print(f'Of those, recoverable: {len(recoverable):,}')
print(f'Unrecoverable (appear without a code): {len(unrecoverable):,}')

Distinct category_ids in null-code rows: 330
Of those, recoverable: 0
Unrecoverable (appear without a code): 330


### Cleaning decisions

Based on the diagnosis:

- **`category_code` (31.5% missing)**: Not recoverable via lookup. The 330 missing-code
  `category_id`s never appear with a code anywhere in the data, so this represents a
  separate, unlabelled tier of the catalogue rather than data quality noise.
  We add a `has_category` flag to enable comparative funnel analysis between the
  labelled and unlabelled populations.
- **`brand` (13.7% missing)**: Same logic — treat as a real "no brand assigned" state.
  When doing brand-level analysis, we filter to non-null rows and report coverage.
- **`user_session` (1 row missing)**: Single glitch row; drop it (sessions are the
  unit of funnel analysis).
- We also split `category_code` into hierarchy levels (`category_l1`, `category_l2`,
  `category_l3`) for easier segmentation.

In [16]:
df = df.dropna(subset=['user_session'])
# Flag for whether the event has a category code
df['has_category'] = df['category_code'].notnull()

# Splitting the dotted category hierarchy into levels
cat_split = df['category_code'].str.split(".", expand=True)
df['category_l1'] = cat_split[0]
df['category_l2'] = cat_split[1] if cat_split.shape[1] > 1 else None
df['category_l3'] = cat_split[2] if cat_split.shape[1] > 2 else None

print(df.shape)
print(df[["category_code", "category_l1", "category_l2", "category_l3", "has_category"]].head())
print(f"\nLabelled events: {df['has_category'].sum():,} ({df['has_category'].mean():.1%})")

print(df['category_l1'].value_counts().head(10))

(8829314, 13)
                         category_code  category_l1  category_l2  \
0                                  NaN          NaN          NaN   
1  appliances.environment.water_heater   appliances  environment   
2           furniture.living_room.sofa    furniture  living_room   
3                   computers.notebook    computers     notebook   
4               electronics.smartphone  electronics   smartphone   

    category_l3  has_category  
0           NaN         False  
1  water_heater          True  
2          sofa          True  
3           NaN          True  
4           NaN          True  

Labelled events: 6,044,500 (68.5%)
category_l1
electronics     3458119
appliances      1050713
computers        507993
auto             269671
apparel          213761
furniture        203312
construction     145148
kids             104879
accessories       45980
sport             37929
Name: count, dtype: int64
